In [1]:
# TASK 3: RuleFlip — Cognitive Flexibility
# CogExec-EF Executive Functions Benchmark
#
# Cognitive faculty : Executive Functions → Cognitive Flexibility
# Research basis    : Wisconsin Card Sorting Task (WCST) adapted for LLMs —
#                     primed on Rule A, then must switch to Rule B
# What it isolates  : Perseveration — continuing to apply Rule A after a
#                     clear switch signal has been issued for Rule B
# Design            : Two-turn conversation (stateful multi-turn)
#                     Turn 1 → apply Rule A (priming)
#                     Turn 2 → switch signal + apply Rule B (test)
#                     switch_type: explicit (125) + implicit (25) = 150 items
# Scoring           : tuple[int, int] → (passes, total_assertions)

import kaggle_benchmarks as kbench
import pandas as pd
import re

df = pd.read_csv("/kaggle/input/datasets/jaytalwar2005/cogexec-ef-benchmark-data/ruleflip_150_FINAL_v3.csv")
print(f"Loaded {len(df)} rows")
print(f"Difficulty:   {df['difficulty'].value_counts().to_dict()}")
print(f"Category:     {df['category'].value_counts().to_dict()}")
print(f"Switch type:  {df['switch_type'].value_counts().to_dict()}")

def safe_pattern(text: str) -> str:
    return re.escape(str(text).strip())

def first_token(text: str) -> str:
    tokens = str(text).strip().split()
    return tokens[0] if tokens else str(text).strip()

Loaded 150 rows
Difficulty:   {'hard': 65, 'medium': 49, 'easy': 36}
Category:     {'mixed_complex': 24, 'classification_shift': 21, 'arithmetic_transform': 18, 'sort_direction': 18, 'alpha_to_semantic': 18, 'pattern_rule': 14, 'filter_change': 11, 'extraction': 10, 'language_transform': 10, 'label_swap': 6}
Switch type:  {'explicit': 125, 'implicit': 25}


In [2]:
from kaggle_benchmarks import llms
import kaggle_benchmarks as kbench

llm1 = kbench.llm   # Gemini Flash (baseline)

llm2 = llms.get("google/gemma-4-26b-a4b")   # weak

llm3 = llms.get("openai/gpt-5.4-mini-2026-03-17")   # mid

llm4 = llms.get("google/gemini-3.1-pro-preview")    # strong

llm5 = llms.get("anthropic/claude-sonnet-4-6@default")   # very strong

llm6 = llms.get("deepseek-ai/deepseek-r1-0528")    # reasoning-heavy

all_models = [llm1, llm2, llm3, llm4, llm5, llm6]

for i, m in enumerate(all_models, 1):
    status = "Loaded" if m else "Failed"
    print(f"llm{i}: {m} --> {status}")

llm1: 🤖 deepseek-ai/deepseek-r1-0528 --> Loaded
llm2: 🤖 google/gemma-4-26b-a4b --> Loaded
llm3: 🤖 openai/gpt-5.4-mini-2026-03-17 --> Loaded
llm4: 🤖 google/gemini-3.1-pro-preview --> Loaded
llm5: 🤖 anthropic/claude-sonnet-4-6@default --> Loaded
llm6: 🤖 deepseek-ai/deepseek-r1-0528 --> Loaded


In [3]:
# ── Task definition ───────────────────────────────────────────────────────────
@kbench.task(name="rule_flip", version=1)
def rule_flip(
    llm,
    id: int,
    items: str,
    rule_a: str,
    rule_a_answer: str,
    rule_b: str,
    rule_b_answer: str,
    switch_signal: str,
    category: str,
    difficulty: str,
    explanation: str,
    switch_type: str,
    answer_format: str,
    scoring_method: str,
    ef_component: str,
) -> tuple[int, int]:
    """Cognitive flexibility: after Rule A priming, fully switch to Rule B without perseverating. Two-turn conversation. switch_type: explicit/implicit. Difficulty: easy/medium/hard."""

    # Turn 1: apply Rule A — primes the model on default behaviour
    llm.prompt(
        f"Apply this rule to the items below.\n\n"
        f"Rule: {rule_a}\n"
        f"Items: {items}\n\n"
        "Reply with ONLY the result. No explanation, no labels."
    )

    # Turn 2: switch signal + Rule B — the real cognitive flexibility test
    # switch_type=explicit: clear verbal signal; switch_type=implicit: subtle shift
    response = llm.prompt(
        f"{switch_signal}\n\n"
        f"New Rule: {rule_b}\n"
        f"Same items: {items}\n\n"
        "Reply with ONLY the result. No explanation, no labels."
    )

    passes = 0
    total = 0

    # Assertion 1: Rule B answer must be present
    total += 1
    r1 = kbench.assertions.assert_contains_regex(
        rf"(?i){safe_pattern(first_token(rule_b_answer))}",
        response,
        expectation=f"Must apply Rule B correctly — expected: '{rule_b_answer}' (switch_type: {switch_type})",
    )
    if r1.passed:
        passes += 1

    # Assertion 2: Rule A answer must NOT dominate (perseveration check)
    a_first = first_token(rule_a_answer)
    b_first = first_token(rule_b_answer)
    if a_first.lower() != b_first.lower():
        total += 1
        r2 = kbench.assertions.assert_not_contains_regex(
            rf"(?i)^{safe_pattern(a_first)}",
            response,
            expectation=(
                f"Must NOT perseverate — Rule A output starts with '{a_first}'. "
                f"Explanation: {explanation}"
            ),
        )
        if r2.passed:
            passes += 1

    return passes, total

In [4]:
# ── Smoke test ────────────────────────────────────────────────────────────────
print("\n── Smoke test (row 0) ──")
smoke = df.iloc[0]
run = rule_flip.run(
    llm=kbench.llm,
    id=int(smoke["id"]),
    items=smoke["items"],
    rule_a=smoke["rule_a"],
    rule_a_answer=smoke["rule_a_answer"],
    rule_b=smoke["rule_b"],
    rule_b_answer=smoke["rule_b_answer"],
    switch_signal=smoke["switch_signal"],
    category=smoke["category"],
    difficulty=smoke["difficulty"],
    explanation=smoke["explanation"],
    switch_type=smoke["switch_type"],
    answer_format=smoke["answer_format"],
    scoring_method=smoke["scoring_method"],
    ef_component=smoke["ef_component"],
)
print(f"Result: {run.result}  |  Passed: {run.passed}")
print("Smoke test complete")


── Smoke test (row 0) ──


Result: (2, 2)  |  Passed: True
Smoke test complete


In [5]:
# ── Multi-model evaluation ────────────────────────────────────────────────────
print("\n── Multi-model evaluation ──")
runs = rule_flip.evaluate(
    llm=all_models,
    evaluation_data=df,
    n_jobs=4,
    max_attempts=3,
    retry_delay=5,
)


── Multi-model evaluation ──


In [6]:
# ── Results ───────────────────────────────────────────────────────────────

results_df = runs.as_dataframe()

# Convert (passes, total) → score
results_df["score"] = results_df["result"].apply(
    lambda x: x[0] / x[1] if isinstance(x, tuple) and x[1] > 0 else float(x)
)

# Clean model names
results_df["model_name"] = results_df["llm"].apply(lambda x: str(x))

# ── BASIC STATS ───────────────────────────────────────────────────────────

print(f"\nTotal runs    : {len(results_df)}")
print(f"Overall score : {results_df['score'].mean():.3f}")

# ── BREAKDOWN ─────────────────────────────────────────────────────────────

print("\nScore by difficulty:")
print(results_df.groupby("difficulty")["score"].mean().round(3))

print("\nScore by category:")
print(results_df.groupby("category")["score"].mean().round(3))

# Only if column exists
if "ef_component" in results_df.columns:
    print("\nScore by ef_component:")
    print(results_df.groupby("ef_component")["score"].mean().round(3))

print("\nScore by model:")
print(results_df.groupby("model_name")["score"].mean().round(3))

# ── MODEL COMPARISON TABLE───────────────────

print("\n── Model comparison (table) ──")

pivot_df = results_df.pivot_table(
    index="id",
    columns="model_name",
    values="score"
)

print(pivot_df.round(3))


results_df.to_csv("final_results.csv", index=False)
pivot_df.to_csv("model_comparison.csv")

print("\nResults saved as CSV files")


Total runs    : 268
Overall score : 0.836

Score by difficulty:
difficulty
easy      0.935
hard      0.764
medium    0.842
Name: score, dtype: float64

Score by category:
category
alpha_to_semantic       0.985
arithmetic_transform    0.983
classification_shift    0.515
extraction              0.944
filter_change           0.929
label_swap              0.417
language_transform      0.929
mixed_complex           0.767
pattern_rule            0.808
sort_direction          0.967
Name: score, dtype: float64

Score by ef_component:
ef_component
cognitive_flexibility_alpha_to_semantic_explicit_switch       0.984
cognitive_flexibility_alpha_to_semantic_implicit_switch       1.000
cognitive_flexibility_arithmetic_transform_explicit_switch    0.981
cognitive_flexibility_arithmetic_transform_implicit_switch    1.000
cognitive_flexibility_classification_shift_explicit_switch    0.481
cognitive_flexibility_classification_shift_implicit_switch    0.667
cognitive_flexibility_extraction_explicit_swit